In [ ]:
# ---------------------------------------------------------------
# CAPÍTULO 5 — Ejercicio 1
# ---------------------------------------------------------------

import seaborn as sns
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

titanic = sns.load_dataset("titanic")
titanic = titanic.dropna(subset=["embarked"])
titanic["age"]     = titanic["age"].fillna(titanic["age"].median())
titanic["sex_cod"] = titanic["sex"].map({"male": 0, "female": 1})
features = ["pclass", "sex_cod", "age"]
X = titanic[features].to_numpy()
y = titanic["survived"].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

model = LogisticRegression(random_state=42)
model.fit(X_train_s, y_train)

y_pred_05 = model.predict(X_test_s)

y_prob     = model.predict_proba(X_test_s)[:, 1]
y_pred_03  = (y_prob >= 0.3).astype(int)

print("=== Umbral 0.5 ===")
print(confusion_matrix(y_test, y_pred_05))
print(classification_report(y_test, y_pred_05))

print("\n=== Umbral 0.3 ===")
print(confusion_matrix(y_test, y_pred_03))
print(classification_report(y_test, y_pred_03))


In [ ]:
# ---------------------------------------------------------------
# CAPÍTULO 5 — Ejercicio 2
# ---------------------------------------------------------------

# Continúa desde la preparación del Ejercicio 1
from sklearn.metrics import roc_auc_score

features_ext = ['pclass', 'sex_cod', 'age', 'fare', 'sibsp']
X_ext = titanic[features_ext].to_numpy()

X_tr2, X_te2, y_tr2, y_te2 = train_test_split(
    X_ext, y, test_size=0.2, random_state=42, stratify=y
)
sc2 = StandardScaler()
X_tr2_s = sc2.fit_transform(X_tr2)
X_te2_s  = sc2.transform(X_te2)

model2 = LogisticRegression(random_state=42)
model2.fit(X_tr2_s, y_tr2)

acc2  = model2.score(X_te2_s, y_te2)
auc2  = roc_auc_score(y_te2,
        model2.predict_proba(X_te2_s)[:, 1])

print(f"Modelo 5 features | Acc: {acc2:.4f} | AUC: {auc2:.4f}")

# pd se declara en el capitulo
coefs = pd.Series(model2.coef_[0], index=features_ext)
print(coefs.sort_values(ascending=False))


In [ ]:
# ---------------------------------------------------------------
# CAPÍTULO 5 — Ejercicio 3
# ---------------------------------------------------------------

import pandas as pd

# Usar el modelo base (3 features) del Ejercicio 1
y_prob = model.predict_proba(X_test_s)[:, 1]

# Reconstruir DataFrame con las features originales + probabilidad
df_test = pd.DataFrame(X_test, columns=features)
df_test["prob_supervivencia"] = y_prob
df_test["sex"] = df_test["sex_cod"].map({0: "male", 1: "female"})

top10 = df_test.sort_values("prob_supervivencia", ascending=False).head(10)
print(top10[["prob_supervivencia", "age", "sex", "pclass"]].round(3))
